In [44]:
from shopping_assistant.env import load_env
load_env()


True

In [9]:
import pandas as pd

In [2]:
from agents import Agent, Runner, function_tool, set_tracing_disabled

set_tracing_disabled(True)

In [56]:
import weaviate
from weaviate.classes.query import Filter
from weaviate import WeaviateClient

In [10]:
prod_df = pd.read_csv("../data/products_extended_w_id.csv")

In [23]:
from collections import defaultdict

def build_nested_dict():
    return defaultdict(build_nested_dict)

def insert_in_tree(tree, hierarchy, item):
    node = tree
    for level in hierarchy[:-1]:
        node = node[item[level]]
    # Last level is a list
    node.setdefault("_items", []).append(item[hierarchy[-1]])

def tree_to_string(tree, indent=0):
    lines = []
    for key in sorted(tree.keys()):
        if key == "_items":
            for v in sorted(tree[key]):
                lines.append("  " * indent + f"- {v}")
        else:
            lines.append("  " * indent + f"{key}")
            lines.extend(tree_to_string(tree[key], indent + 1))
    return lines

def get_hierarchy_tree(data: list[dict], hierarchy: list) -> str:
    tree = build_nested_dict()
    
    for item in data:
        insert_in_tree(tree, hierarchy, item)

    return "\n".join(tree_to_string(tree))


In [24]:
prod_hier_dict = prod_df[['category', 'subcategory']].drop_duplicates().to_dict('records')
prod_hier_dict

[{'category': 'jewelry', 'subcategory': 'bracelet'},
 {'category': 'jewelry', 'subcategory': 'earrings'},
 {'category': 'jewelry', 'subcategory': 'necklace'},
 {'category': 'apparel', 'subcategory': 'dress'},
 {'category': 'footwear', 'subcategory': 'shoes'},
 {'category': 'apparel', 'subcategory': 'skirt'},
 {'category': 'accessories', 'subcategory': 'bag'},
 {'category': 'accessories', 'subcategory': 'sunglasses'},
 {'category': 'apparel', 'subcategory': 'top blouse sweater'}]

In [26]:
print(get_hierarchy_tree(prod_hier_dict, ["category", "subcategory"]))

accessories
  - bag
  - sunglasses
apparel
  - dress
  - skirt
  - top blouse sweater
footwear
  - shoes
jewelry
  - bracelet
  - earrings
  - necklace


In [27]:
PRODUCT_CATALOG_HIERARCHY_TREE = get_hierarchy_tree(prod_hier_dict, ["category", "subcategory"])

In [29]:
PRODUCT_CATALOG_PROMPT_DESCRIPTION = f"""
The product catalog hierarchy is described as follows: 

Template:
```
<category>
    <subcategory>
```

Values:
```
{PRODUCT_CATALOG_HIERARCHY_TREE}
```
"""

print(PRODUCT_CATALOG_PROMPT_DESCRIPTION)


The product catalog hierarchy is described as follows: 

Template:
```
<category>
    <subcategory>
```

Values:
```
accessories
  - bag
  - sunglasses
apparel
  - dress
  - skirt
  - top blouse sweater
footwear
  - shoes
jewelry
  - bracelet
  - earrings
  - necklace
```



In [90]:
from pydantic import BaseModel, Field

class ProductInfoParserOutput(BaseModel):
    category: str | None = Field(description="Category of the product")
    subcategory: str | None = Field(description="Subcategory of the product")
    min_price: float | None = Field(description="Minimum price of the product")
    max_price: float | None = Field(description="Maximum price of the product")
    retrieval_query: str | None = Field(description="Rephrased user query suited for direct product retrieval. DO NOT populate if clarification is needed.")
    product_clarification_response: str | None = Field(description="Response to clarify the product.")



In [162]:
PRODUCT_INFO_PARSER_SYSTEM_PROMPT = """
You are a shopping assistant, who helps users find products in an e-commerce store.
You are given the product catalog hierarchy as follows:
{catalog_hierarchy_tree}

You previously recommended the following products to the user:

{prev_recommended_products}

You will be given a user query. 
Follow the steps below and ALWAYS keep in mind the previous products you recommended to refine your response.

STEPS:

1. Identify the category from the user query => `category`

If no category is mentioned, return None and proceed to step 2, else jump to step 3

2A.  Draft a clarification response as a followup to user as `product_clarification_response`. NOW EXIT and populate rest of the fields as None.

2B. Identify the subcategory from the user query => `subcategory`

If no subcategory is mentioned, return None.

3. Identify the price range (min_price, max_price) from the user query => `price_range`

If no price range is mentioned, return None.

4. Rephrase the user query to a more specific product search query => `retrieval_query`
""".format(
    catalog_hierarchy_tree=PRODUCT_CATALOG_PROMPT_DESCRIPTION, prev_recommended_products='{prev_recommended_products}'
)

In [163]:
print(PRODUCT_INFO_PARSER_SYSTEM_PROMPT)


You are a shopping assistant, who helps users find products in an e-commerce store.
You are given the product catalog hierarchy as follows:

The product catalog hierarchy is described as follows: 

Template:
```
<category>
    <subcategory>
```

Values:
```
accessories
  - bag
  - sunglasses
apparel
  - dress
  - skirt
  - top blouse sweater
footwear
  - shoes
jewelry
  - bracelet
  - earrings
  - necklace
```


You previously recommended the following products to the user:

{prev_recommended_products}

You will be given a user query. 
Follow the steps below and ALWAYS keep in mind the previous products you recommended to refine your response.

STEPS:

1. Identify the category from the user query => `category`

If no category is mentioned, return None and proceed to step 2, else jump to step 3

2A.  Draft a clarification response as a followup to user as `product_clarification_response`. NOW EXIT and populate rest of the fields as None.

2B. Identify the subcategory from the user quer

In [164]:
import openai

openai_client = openai.OpenAI()

In [165]:
input_messages = [
    {"role": "system", "content": PRODUCT_INFO_PARSER_SYSTEM_PROMPT},
    {"role": "user", "content": "I am looking for some leather bags in black color, ab bit funky. My budget is around $100"}
]
response = openai_client.responses.parse(
    input=input_messages,
    text_format=ProductInfoParserOutput,
    model="gpt-4o-mini"
)

response.output_parsed.model_dump()

{'category': 'accessories',
 'subcategory': 'bag',
 'min_price': None,
 'max_price': 100.0,
 'retrieval_query': 'black funky leather bags under $100',
 'product_clarification_response': None}

In [166]:
input_messages = [
    {"role": "system", "content": PRODUCT_INFO_PARSER_SYSTEM_PROMPT},
    {"role": "user", "content": "I am browsing for some products which suit my style in black"}
]
response = openai_client.responses.parse(
    input=input_messages,
    text_format=ProductInfoParserOutput,
    model="gpt-4o-mini"
)

response.output_parsed.model_dump()

{'category': None,
 'subcategory': None,
 'min_price': None,
 'max_price': None,
 'retrieval_query': None,
 'product_clarification_response': 'Could you specify what type of products you are looking for, such as clothing, accessories, or jewelry?'}

In [167]:

import functools

def _retrieve_products_client(
    client,
    collection_name,
    query,
    filter_obj,
    limit
):
    """Retrieve products based on query, category, subcategory, and price range."""

    products = client.collections.use(collection_name)
    responses = products.query.near_text(
            query=query,
            filters=filter_obj,
            limit=limit
        )

    return responses


def retrieve_products(
    query: str = "",
    categories: list[str] | None = None,
    subcategories: list[str] | None = None,
    min_price: float | None = None,
    max_price: float | None = None,
    limit: int = 5,
    client: WeaviateClient | None = None,
    collection_name: str = "products"
):
    """Retrieve products based on query, category, subcategory, and price range."""

    filters = []

    if categories:
        filters.append(
            Filter.by_property("category").contains_any(categories)
        )

    if subcategories:
        filters.append(
            Filter.by_property("subcategory").contains_any(subcategories)
        )

    if min_price:
        filters.append(
            Filter.by_property("price").greater_or_equal(min_price)
        )

    if max_price:
        filters.append(
            Filter.by_property("price").less_or_equal(max_price)
        )

    
    filter_obj = functools.reduce(
        lambda x, y: x & y,
        filters
    )

    if client is None:
        with weaviate.connect_to_local() as client:
            responses = _retrieve_products_client(
                client,
                collection_name,
                query,
                filter_obj,
                limit
            )
    else:
        responses = _retrieve_products_client(
            client,
            collection_name,
            query,
            filter_obj,
            limit
        )

    return responses
    

In [168]:
class Product(BaseModel):
    product_id: int
    name: str
    description: str
    category: str
    subcategory: str
    price: float

class ProductRetrievalAgentWorkflowResponse(BaseModel):
    response: str
    products_retrieved: list[Product] = None
    success: bool


def get_product_list_prompt_str(products: list[Product]):
    prod_desc_lst = []
    for idx, prod in enumerate(products):
        prod_desc = f"{idx + 1}. {prod.name} (${prod.price})"
        prod_desc_lst.append(prod_desc)    
    return "\n".join(prod_desc_lst)

def product_retrieval_agent_workflow(
    user_query: str,
    client: WeaviateClient = None,
    n: int = 5,
    history: list[dict] = None,
    prev_recommended_products: list[Product] = None
):

    if history is None:
        history = []

    if prev_recommended_products is None:
        prev_recommended_products = []
        prev_recommended_products_str = "<EMPTY LIST>: No products were recommended earlier."
    else:
        prev_recommended_products_str = get_product_list_prompt_str(prev_recommended_products)

    input_messages = [
        {"role": "system", "content": PRODUCT_INFO_PARSER_SYSTEM_PROMPT.format(prev_recommended_products=prev_recommended_products_str)},
        *history,
        {"role": "user", "content": user_query}
    ]
    response = openai_client.responses.parse(
        input=input_messages,
        text_format=ProductInfoParserOutput,
        model="gpt-4o-mini"
    )

    product_info_parsed: ProductInfoParserOutput = response.output_parsed

    if product_info_parsed.product_clarification_response:

        return ProductRetrievalAgentWorkflowResponse(
            response=product_info_parsed.product_clarification_response,
            success=False
        )


    product_retrieval_response = retrieve_products(
        query=product_info_parsed.retrieval_query,
        categories=[product_info_parsed.category] if product_info_parsed.category else None,
        subcategories=[product_info_parsed.subcategory] if product_info_parsed.subcategory else None,
        min_price=product_info_parsed.min_price,
        max_price=product_info_parsed.max_price,
        client=client,
        limit=n
    )

    products_retrieved = [Product(**p.properties) for p in product_retrieval_response.objects]

    product_list_str = get_product_list_prompt_str(products_retrieved)

    response = "Would you like to explore any of the following products?\n\n"
    response += product_list_str
    
    return ProductRetrievalAgentWorkflowResponse(
        response=response,
        products_retrieved=products_retrieved,
        success=True
    )



    

In [170]:
products_retrieval_agent_resp = product_retrieval_agent_workflow(
    user_query="I am browsing for some black leather bags.",
    n=5,
    history=[],
    prev_recommended_products=None
)

In [171]:
print(products_retrieval_agent_resp.response)

Would you like to explore any of the following products?

1. Classic Black Patent Leather Purse ($49.9)
2. Sleek Leather Satchel Bag ($49.99)
3. Opulent Leather Tote Bag ($99.99)
4. Navy Leather Everyday Bag ($139.99)
5. Fusion Leather Crossbody Bag ($109.99)


In [172]:
history = [
    {"role": "user", "content": "I am browsing for some black leather bags."},
    {"role": "assistant", "content": products_retrieval_agent_resp.response}
]

In [174]:
r = product_retrieval_agent_workflow(
    user_query="Can you show more like black patent leather one?",
    n=5,
    history=history,
    prev_recommended_products=products_retrieval_agent_resp.products_retrieved,
)

In [176]:
print(r.response)

Would you like to explore any of the following products?

1. Classic Black Patent Leather Purse ($49.9)
2. Embroidered Leather Bohemian Purse ($49.9)
